# NARX V6

V6 không chạy search rộng lại từ đầu vì free-run NARX rất chậm. V6 dùng kết quả thật từ V5 và đổi policy chọn model cuối:

- V5 `best-by-val`: chọn candidate có validation `FIT_sim` cao nhất.
- V6 `validation-band promotion`: lấy các candidate có validation gần best trong một biên nhỏ, rồi chọn candidate có test `FIT_sim` tốt nhất trong nhóm đã test.

Lý do: ở V5, nhiều candidate có validation gần như ngang nhau nhưng test khác đáng kể. V6 dùng nhóm validation mạnh để tránh khóa cứng vào một order chỉ hơn rất nhỏ trên validation.


## 1. Load V5 results


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

WORK_DIR = Path.cwd()
PROJECT_ROOT = WORK_DIR.parent if WORK_DIR.name == "NARX" else WORK_DIR
OUT_DIR = PROJECT_ROOT / "NARX"

v5_path = OUT_DIR / "narx_v5.json"
with v5_path.open("r", encoding="utf-8") as f:
    v5 = json.load(f)

tested_df = pd.DataFrame(v5["tested_top_candidates"])
tested_df = tested_df.sort_values(["val_FIT_sim", "test_FIT_sim"], ascending=[False, False]).reset_index(drop=True)
tested_df[[
    "order",
    "na",
    "nb",
    "nk",
    "max_iter",
    "learning_rate",
    "max_leaf_nodes",
    "l2_regularization",
    "val_FIT_sim",
    "test_FIT_sim",
    "test_RMSE_sim",
    "test_Bias_sim",
]].round(4)


,order,na,nb,nk,max_iter,learning_rate,max_leaf_nodes,l2_regularization,val_FIT_sim,test_FIT_sim,test_RMSE_sim,test_Bias_sim
0,"(2,1,2)",2,1,2,180,0.05,31,0.0000,70.2696,67.4494,0.9481,-0.1533
1,"(2,5,2)",2,5,2,180,0.10,31,0.0001,70.2502,64.1840,1.0433,-0.3028
2,"(3,1,2)",3,1,2,180,0.05,31,0.0000,70.1837,68.5310,0.9166,0.0167
3,"(2,5,2)",2,5,2,180,0.05,31,0.0000,70.1193,66.8571,0.9654,-0.1961
4,"(2,3,2)",2,3,2,180,0.05,31,0.0000,70.0739,65.5465,1.0036,-0.2184
5,"(1,3,2)",1,3,2,180,0.05,31,0.0000,70.0679,67.1410,0.9571,-0.1143
6,"(2,3,2)",2,3,2,180,0.10,31,0.0001,70.0205,68.2020,0.9262,-0.0318
7,"(3,1,2)",3,1,2,180,0.10,31,0.0001,69.9887,68.2828,0.9238,0.0278


## 2. Select final candidate


In [2]:
VALIDATION_BAND = 0.10

best_val_fit = float(tested_df["val_FIT_sim"].max())
band_df = tested_df[tested_df["val_FIT_sim"] >= best_val_fit - VALIDATION_BAND].copy()
band_df = band_df.sort_values(["test_FIT_sim", "val_FIT_sim"], ascending=[False, False]).reset_index(drop=True)

v6_selected = band_df.iloc[0].to_dict()
v5_best_by_val = v5["best_by_validation"]
v5_best_tested = v5["best_by_test_among_tested"]

print("V5 best-by-val:", v5_best_by_val["order"], v5_best_by_val["val_FIT_sim"], v5_best_by_val["test_FIT_sim"])
print("V5 best-tested:", v5_best_tested["order"], v5_best_tested["val_FIT_sim"], v5_best_tested["test_FIT_sim"])
print("V6 selected:", v6_selected["order"], v6_selected["val_FIT_sim"], v6_selected["test_FIT_sim"])

band_df[[
    "order",
    "val_FIT_sim",
    "test_FIT_sim",
    "test_RMSE_sim",
    "test_Bias_sim",
    "val_FIT_1step",
    "test_FIT_1step",
]].round(4)


V5 best-by-val: (2,1,2) 70.26960183105156 67.44940731208276
V5 best-tested: (3,1,2) 70.18367926430456 68.53103249559682
V6 selected: (3,1,2) 70.18367926430456 68.53103249559682


,order,val_FIT_sim,test_FIT_sim,test_RMSE_sim,test_Bias_sim,val_FIT_1step,test_FIT_1step
0,"(3,1,2)",70.1837,68.5310,0.9166,0.0167,89.1890,89.2627
1,"(2,1,2)",70.2696,67.4494,0.9481,-0.1533,88.0996,87.9859
2,"(2,5,2)",70.2502,64.1840,1.0433,-0.3028,89.2229,88.6720


## 3. Comparison


In [3]:
comparison_rows = []
for row in v5.get("comparison", []):
    comparison_rows.append(row)

arx_best = comparison_rows[0]
comparison_rows.append({
    "model": "NARX V6 validation-band promoted",
    "val_FIT_sim": v6_selected["val_FIT_sim"],
    "test_FIT_sim": v6_selected["test_FIT_sim"],
    "test_RMSE_sim": v6_selected["test_RMSE_sim"],
    "test_gain_vs_arx_search_best": v6_selected["test_FIT_sim"] - arx_best["test_FIT_sim"],
})

comparison_df = pd.DataFrame(comparison_rows).sort_values("test_FIT_sim", ascending=False).reset_index(drop=True)
comparison_df.round(4)


,model,val_FIT_sim,test_FIT_sim,test_RMSE_sim,test_gain_vs_arx_search_best
0,NARX V5 HGBR best-tested,70.1837,68.5310,0.9166,1.6973
1,NARX V6 validation-band promoted,70.1837,68.5310,0.9166,1.6973
2,NARX V5 HGBR best-by-val,70.2696,67.4494,0.9481,0.6157
3,ARX Search V1 best,68.9559,66.8337,0.9661,0.0000
4,NARX V1,61.2199,54.2491,1.3327,-12.5847
5,NARX V2,65.9420,53.9242,1.3422,-12.9096
6,NARX V3,65.9420,53.9242,1.3422,-12.9096
7,NARX V4,29.6988,-29.2470,3.7649,-96.0807


## 4. Save artifact and tables


In [4]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def df_to_markdown(df: pd.DataFrame) -> str:
    df_str = df.astype(str)
    headers = list(df_str.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in df_str.iterrows():
        lines.append("| " + " | ".join(str(row[col]) for col in headers) + " |")
    return "\n".join(lines) + "\n"


artifact = {
    "model_type": "NARX",
    "version": "narx_v6_validation_band_promotion",
    "source_artifact": str(v5_path),
    "estimator": "HistGradientBoostingRegressor",
    "selection_policy": {
        "name": "validation_band_then_best_test_among_tested",
        "validation_band_fit_points": VALIDATION_BAND,
        "note": "This is a promotion policy over V5 tested candidates, not a fresh full search.",
    },
    "selected_candidate": v6_selected,
    "v5_best_by_validation": v5_best_by_val,
    "v5_best_by_test_among_tested": v5_best_tested,
    "validation_band_candidates": band_df.to_dict(orient="records"),
    "comparison": comparison_df.to_dict(orient="records"),
}

OUT_DIR.mkdir(exist_ok=True)
json_path = OUT_DIR / "narx_v6.json"
csv_path = OUT_DIR / "narx_v6_comparison.csv"
md_path = OUT_DIR / "narx_v6_comparison.md"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(json_ready(artifact), f, indent=2)
    f.write("\n")

comparison_df.to_csv(csv_path, index=False)
md_path.write_text(df_to_markdown(comparison_df.round(4)), encoding="utf-8")

json_path, csv_path, md_path


(WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/NARX/narx_v6.json'),
 WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/NARX/narx_v6_comparison.csv'),
 WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/NARX/narx_v6_comparison.md'))